# MMUbiPred–Context Residual v1 — fold-safe development benchmark

This notebook retrieves and validates full UniProt sequences, then generates protein-grouped out-of-fold predictions from two experts: the paper-compatible 49-residue MMUbiPred topology and a target-aware rank-8 LoRA ESM-2 expert over 257 residues. A nonnegative residual stacker is evaluated by an additional cross-fit. Development and refitting do not load the released independent test. The final guarded section performs one frozen historical-test evaluation after full-data refitting. Select a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Add a fine-grained read-only GitHub token to Colab Secrets as `GITHUB_TOKEN`. Adding `HF_TOKEN` is recommended for the ESM-2 download.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN, HF_TOKEN = secret('GITHUB_TOKEN'), secret('HF_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/content/drive/MyDrive/MMUbiPred-research/huggingface-cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json, torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Preflight tests

In [ ]:
run_live([sys.executable, '-u', '-m', 'unittest', 'discover', '-s', 'experiment/tests'])

## Retrieve and validate full-protein context

The cache is stored on Drive and is updated after every UniProt request batch. Rerunning this cell retrieves only missing accessions. A site is eligible only when UniProt reproduces its released 49-mer exactly.

In [ ]:
CONTEXT_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/context')
CONTEXT_ROOT.mkdir(parents=True, exist_ok=True)
SEQUENCE_CACHE = CONTEXT_ROOT / 'uniprot_training_sequences.json'
VALIDATION_REPORT = CONTEXT_ROOT / 'context_validation.json'
run_live([
    sys.executable, '-u', 'experiment/scripts/fetch_uniprot_context.py',
    '--data-dir', 'replication/MMUbiPred',
    '--output', SEQUENCE_CACHE,
    '--validation-report', VALIDATION_REPORT,
    '--context-window-size', '257',
])
validation = json.loads(VALIDATION_REPORT.read_text())
display(validation['context_validation'])

## Run resumable outer folds

Each fold trains the local expert first and the ESM-2 context expert second. Keep `FOLDS_TO_RUN = [0]` for one fold per Colab session, or list several folds if enough runtime remains. Completed expert predictions are reused; interrupted training resumes after its latest completed epoch.

In [ ]:
RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-v1')
FOLDS_TO_RUN = [0]  # Change to [1], [2], [3], then [4] in later sessions.

command = [
    sys.executable, '-u', 'experiment/scripts/run_context_residual_cv.py',
    '--config', 'experiment/configs/mmubipred_context_residual_v1.json',
    '--output-dir', RUN_DIR,
    '--resume',
]
for fold in FOLDS_TO_RUN:
    command.extend(['--fold', str(fold)])
run_live(command)

## Inspect the completed OOF comparison

This cell becomes available automatically after both experts finish all five folds. `residual_stack_crossfit` is the honest stacked development estimate; the full-OOF fitted weights are retained only for a later frozen refit.

In [ ]:
SUMMARY_PATH = RUN_DIR / 'summary.json'
if not SUMMARY_PATH.exists():
    completed = sorted(str(path.relative_to(RUN_DIR)) for path in RUN_DIR.glob('folds/fold_*/*/outer_predictions.npz'))
    print(f'Completed expert-fold predictions: {len(completed)}/10')
    display(completed)
else:
    summary = json.loads(SUMMARY_PATH.read_text())
    display(summary['metrics'])
    display(summary['stack_minus_local_fixed_threshold'])
    print('Predeclared decision:', summary['predeclared_decision'])
    display(summary['predeclared_decision_checks'])
    display(summary['crossfit_stackers'])
    display(summary['final_stacker_for_future_refit'])
    print('Independent test accessed:', summary['independent_test_accessed'])

## Protocol amendment: stabilize the local expert only

Run this section only after all five original context folds are complete. Original local fold 4 predicted one class, so the original automatic `GO` is invalid. This amendment preserves every expensive context prediction. For each fold it trains the inexpensive local model with initialization seeds 42, 123, and 2026, rejects inner-validation single-class predictions, and selects by inner fixed-threshold MCC without accessing the outer fold. Run one or several folds per session.

In [ ]:
LOCAL_FOLDS_TO_RUN = [0]  # Then [1], [2], [3], and [4], or combine if runtime permits.

command = [
    sys.executable, '-u',
    'experiment/scripts/stabilize_context_residual_local.py',
    '--config', 'experiment/configs/mmubipred_context_residual_v1.json',
    '--output-dir', RUN_DIR,
    '--resume',
]
for fold in LOCAL_FOLDS_TO_RUN:
    command.extend(['--fold', str(fold)])
run_live(command)

## Inspect the corrected comparison

The corrected summary is created after all five stabilized local folds exist. A collapsed outer expert fold now makes the comparison invalid automatically.

In [ ]:
STABILIZED_SUMMARY_PATH = RUN_DIR / 'stabilized_summary.json'
if not STABILIZED_SUMMARY_PATH.exists():
    completed = sorted(
        str(path.relative_to(RUN_DIR))
        for path in RUN_DIR.glob(
            'folds/fold_*/local_stabilized/outer_predictions.npz'
        )
    )
    print(f'Completed stabilized local folds: {len(completed)}/5')
    display(completed)
else:
    stabilized = json.loads(STABILIZED_SUMMARY_PATH.read_text())
    print('Comparison valid:', stabilized['comparison_valid'])
    print('Decision:', stabilized['predeclared_decision'])
    print('Candidate decision:', stabilized['amended_candidate_decision'])
    display(stabilized['collapsed_expert_folds'])
    display(stabilized['metrics'])
    display(stabilized['stack_minus_local_fixed_threshold'])
    display(stabilized['context_minus_local_fixed_threshold'])
    display(stabilized['stack_minus_context_fixed_threshold'])
    display(stabilized['predeclared_decision_checks'])
    display(stabilized['amended_context_advancement_checks'])
    display(stabilized['amended_fusion_advancement_checks'])
    display(stabilized['crossfit_stackers'])
    display(stabilized['final_stacker_for_future_refit'])
    print('Independent test accessed:', stabilized['independent_test_accessed'])

## Frozen context-only full-data refit

Run this only after the corrected comparison reports `ADVANCE_CONTEXT_ONLY`. The five context best epochs `[8, 8, 7, 6, 7]` freeze the duration at their integer median: **7 epochs**. A fresh context-only LoRA-ESM2 model is trained on all 89,551 released training records that passed exact UniProt-context validation. There is no validation, early stopping, threshold search, or test access. The run saves an exact resume checkpoint after every completed epoch, so rerunning this cell continues from the next unfinished epoch.

In [ ]:
RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-v1')
FINAL_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')

run_live([
    sys.executable, '-u',
    'experiment/scripts/refit_context_candidate.py',
    '--config', 'experiment/configs/mmubipred_context_residual_v1.json',
    '--development-run-dir', str(RUN_DIR),
    '--output-dir', str(FINAL_RUN_DIR),
    '--resume',
])

## Inspect refit progress or completion

A completed run has seven history entries and a compact `best.pt`. These are training artifacts only; this notebook still does not load the released independent test.

In [ ]:
REFIT_HISTORY_PATH = FINAL_RUN_DIR / 'refit_history.json'
REFIT_SUMMARY_PATH = FINAL_RUN_DIR / 'refit_summary.json'
if REFIT_HISTORY_PATH.exists():
    refit_history = json.loads(REFIT_HISTORY_PATH.read_text())
    print(f'Completed refit epochs: {len(refit_history)}/7')
    display(refit_history)
if REFIT_SUMMARY_PATH.exists():
    display(json.loads(REFIT_SUMMARY_PATH.read_text()))
else:
    print('The frozen refit is not complete yet.')
print('Released independent test loaded by this notebook:', False)

## Guarded historical-test context retrieval

Run this only for the completed frozen seven-epoch checkpoint. It accesses the released independent-test identifiers, retrieves their UniProt sequences into a separate resumable Drive cache, and records exact context-validation coverage. It does not train or modify the model.

In [ ]:
TEST_CONTEXT_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/context')
TEST_CONTEXT_ROOT.mkdir(parents=True, exist_ok=True)
TEST_SEQUENCE_CACHE = TEST_CONTEXT_ROOT / 'uniprot_test_sequences.json'
TEST_VALIDATION_REPORT = TEST_CONTEXT_ROOT / 'test_context_validation.json'

run_live([
    sys.executable, '-u', 'experiment/scripts/fetch_uniprot_context.py',
    '--data-dir', 'replication/MMUbiPred',
    '--split', 'test',
    '--allow-locked-test',
    '--output', str(TEST_SEQUENCE_CACHE),
    '--validation-report', str(TEST_VALIDATION_REPORT),
    '--context-window-size', '257',
])
test_validation = json.loads(TEST_VALIDATION_REPORT.read_text())
display(test_validation['context_validation'])

## One-time frozen historical-test evaluation

This reconstructs the pretrained ESM-2 backbone, loads only the frozen epoch-7 LoRA and classifier parameters, and evaluates threshold `0.5`. Existing result files prevent an accidental repeat. A direct comparison with the paper's full-cohort numbers is marked valid only if every processed test record passes context validation; otherwise MMUbiPred must later be evaluated on the saved matched indices.

In [ ]:
FINAL_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')
CONTEXT_TEST_METRICS_PATH = FINAL_RUN_DIR / 'context_test_metrics.json'

if CONTEXT_TEST_METRICS_PATH.exists():
    print('Existing frozen test result found; evaluation was not repeated.')
else:
    run_live([
        sys.executable, '-u',
        'experiment/scripts/evaluate_context_candidate.py',
        '--config', 'experiment/configs/mmubipred_context_residual_v1.json',
        '--run-dir', str(FINAL_RUN_DIR),
        '--sequence-cache', str(TEST_SEQUENCE_CACHE),
        '--batch-size', '32',
        '--num-workers', '0',
        '--allow-locked-test',
    ])

context_test = json.loads(CONTEXT_TEST_METRICS_PATH.read_text())
display(context_test['context_validation'])
display(context_test['fixed_threshold'])
print('Direct full-cohort paper comparison valid:', context_test['direct_full_cohort_paper_comparison_valid'])
display(context_test['fixed_threshold_difference_from_full_paper_baseline'])
print(context_test['matched_comparison_note'])

## Matched-cohort comparison with the released MMUbiPred model

The context model retained 12,288 of 12,598 processed test records, so its metrics cannot be compared directly with the paper's full-cohort aggregate. This final section loads the already-replicated authors' H5 model, first requires exact reproduction of its full-test confusion matrix and metrics, and then evaluates both frozen models on the identical 12,288 saved indices. Installing legacy Keras may print dependency-conflict warnings; no further PyTorch training or inference is needed in this runtime.

In [ ]:
%pip install -q "tf-keras==2.19.0"

In [ ]:
RELEASED_MODEL_PATH = Path(
    '/content/drive/MyDrive/MMUbiPred-replication/artifacts/'
    'DeepUBI_AAindex_One_Hot_Emb_drop_out2423.h5'
)
FINAL_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')
MATCHED_COMPARISON_PATH = FINAL_RUN_DIR / 'matched_mmubipred_comparison.json'

assert RELEASED_MODEL_PATH.exists(), f'Model not found: {RELEASED_MODEL_PATH}'
if MATCHED_COMPARISON_PATH.exists():
    print('Existing matched comparison found; released-model inference was not repeated.')
else:
    run_live([
        sys.executable, '-u',
        'experiment/scripts/evaluate_matched_mmubipred.py',
        '--model-file', str(RELEASED_MODEL_PATH),
        '--context-run-dir', str(FINAL_RUN_DIR),
        '--data-dir', 'replication/MMUbiPred',
        '--batch-size', '32',
    ])

matched = json.loads(MATCHED_COMPARISON_PATH.read_text())
print('Matched comparison valid:', matched['comparison_valid'])
display(matched['paper_model_full_test_verification'])
display(matched['paper_model_matched_test'])
display(matched['context_model_matched_test'])
display(matched['context_minus_paper_matched'])